# Phase 1: Financial Risk Dataset Generation

In order to fine-tune our LLM (Llama-3), we need an **Instruction-Tuning Dataset**.

This script generates a synthetic dataset of 15,000 records containing corporate risk narratives. The goal is to train the LLM to read a narrative (the input) and output a structured JSON response classifying the **Risk Category, Risk Summary, and Severity**.

In [2]:
import json
import random
import pandas as pd
from typing import List, Dict
import os

# Sample vocabulary for risk narratives
COMPANIES = ["Acme Corp", "Globex", "Initech", "Umbrella Corp", "Wayne Enterprises", "Stark Industries", "Massive Dynamic"]
ACTIONS = ["failed to report", "misclassified", "experienced a breach in", "identified anomalies in", "delayed the submission of", "discovered unauthorized access to"]
DOMAINS = ["Q3 financial statements", "user data privacy protocols", "anti-money laundering (AML) controls", "supply chain vendor contracts", "employee compliance training logs", "tax withholding calculations"]
CONSEQUENCES = ["leading to potential regulatory fines.", "causing a 15% drop in stock price.", "resulting in a formal SEC investigation.", "creating significant operational downtime.", "exposing sensitive customer information.", "violating internal governance policies."]

CATEGORIES = ["Financial", "Compliance", "Operational", "Strategic"]
SEVERITIES = ["Low", "Medium", "High"]


In [3]:
def generate_synthetic_record() -> Dict:
    company = random.choice(COMPANIES)
    action = random.choice(ACTIONS)
    domain = random.choice(DOMAINS)
    consequence = random.choice(CONSEQUENCES)

    narrative = f"{company} {action} {domain}, {consequence}"

    # Simple heuristic for category/severity for the dummy dataset
    category = "Compliance"
    if "financial" in domain or "tax" in domain or "stock" in consequence:
        category = "Financial"
    elif "breach" in action or "downtime" in consequence:
        category = "Operational"

    severity = "Medium"
    if "SEC" in consequence or "fine" in consequence or "breach" in action:
        severity = "High"
    elif "delayed" in action or "training" in domain:
        severity = "Low"

    summary = f"Issue with {domain} causing {consequence.strip('.')}"

    return {
        "narrative": narrative,
        "output": json.dumps({
            "Risk Category": category,
            "Risk Summary": summary,
            "Severity": severity
        })
    }

### Execute and Save
We will generate 15,000 records and save them to `data/financial_risk_dataset.jsonl`.

In [4]:
def generate_dataset(num_records: int = 15000, output_path: str = "data/financial_risk_dataset.jsonl"):
    os.makedirs("data", exist_ok=True)
    print(f"Generating {num_records} synthetic risk narrative records...")
    records = [generate_synthetic_record() for _ in range(num_records)]

    with open(output_path, 'w') as f:
        for record in records:
            f.write(json.dumps(record) + "\n")

    print(f"Dataset saved to {output_path}")

# Run the generation
generate_dataset()

Generating 15000 synthetic risk narrative records...
Dataset saved to data/financial_risk_dataset.jsonl


### Preview the Data
Let's look at the first 5 records we generated.

In [5]:
df = pd.read_json("data/financial_risk_dataset.jsonl", lines=True)
df.head()

,narrative,output
0,Wayne Enterprises delayed the submission of em...,"{""Risk Category"": ""Compliance"", ""Risk Summary""..."
1,Initech identified anomalies in supply chain v...,"{""Risk Category"": ""Compliance"", ""Risk Summary""..."
2,Massive Dynamic misclassified employee complia...,"{""Risk Category"": ""Operational"", ""Risk Summary..."
3,Globex delayed the submission of employee comp...,"{""Risk Category"": ""Compliance"", ""Risk Summary""..."
4,Wayne Enterprises failed to report anti-money ...,"{""Risk Category"": ""Compliance"", ""Risk Summary""..."


# Phase 2: LoRA Fine-Tuning Llama-3

This notebook is specifically designed for **Google Colab (T4 GPU)**.
It implements Parameter-Efficient Fine-Tuning (PEFT) using Low-Rank Adaptation (LoRA) and 4-bit quantization (bitsandbytes). This reduces the memory footprint of an 8-Billion parameter model by over 60%, allowing it to train comfortably on a free 16GB GPU.

### First, Install Dependencies

In [6]:
!pip install -q bitsandbytes "transformers<4.45.0" peft accelerate datasets "trl<0.12.0" liger-kernel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 15.8 MB/s eta 0:00:00


### 1. Define the Instruction Prompt Template

In [7]:
def get_risk_classification_prompt(narrative: str) -> str:
    """
    Returns the prompt template aligned with enterprise risk classification workflows.
    """
    return f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an expert financial compliance officer. Analyze the following risk narrative and extract the 'Risk Category' (e.g., Operational, Financial, Compliance, Strategic), a brief 'Risk Summary', and the 'Severity' (High, Medium, Low). Return the output as structured JSON.

### Input:
{narrative}

### Response:
"""

### 2. Load the Dataset
Upload the `financial_risk_dataset.jsonl` file you generated in the previous step into the Colab file explorer.

In [9]:
from datasets import load_dataset

def format_instruction(example):
    prompt = get_risk_classification_prompt(example['narrative'])
    return {"text": f"{prompt}{example['output']}"}

print("Loading dataset...")
dataset = load_dataset("json", data_files="data/financial_risk_dataset.jsonl", split="train")
dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"].map(format_instruction)
eval_dataset = dataset["test"].map(format_instruction)
print("Dataset prepared!")

Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/13500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Dataset prepared!


### 3. Load Model with 4-Bit Quantization
We use `unsloth/llama-3-8b-Instruct-bnb-4bit` because it is already optimized and doesn't require a HuggingFace login token.

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"

# 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:174: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


RuntimeError: No GPU found. A GPU is needed for quantization.

### 4. Configure LoRA (Low-Rank Adaptation)

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

### 5. Execute Fine-Tuning

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="adamw_torch",
    save_steps=200,
    logging_steps=50,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    max_grad_norm=0.3,
    max_steps=100, # Set to higher for full training
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_arguments,
)

print("Starting fine-tuning...")
trainer.train()

### 6. Save the Fine-Tuned Weights
Zip the `finetuned_model_lora` folder and download it to your Mac.

In [ ]:
output_dir = "./finetuned_model_lora"
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model adapters saved to {output_dir}")

!zip -r finetuned_model_lora.zip finetuned_model_lora/